# Pre-processing with Spark

### Dataset: `iris.csv` (150 righe con 50 esemplari ciascuna)
> **Nota metodologica**: Questo progetto ha uno scopo esclusivamente didattico. Per un dataset così piccolo come quello di iris.csv, l'utilizzo di un motore distribuito come Spark è tecnicamente sproporzionato.

> Questo esercizio ha l'obiettivo di offrire un apprendimento pratico e approfondito degli strumenti e della logica del framework Spark. Fornisce inoltre un'introduzione al Machine Learning in PySpark per esplorare i piani di esecuzione fisici e comprendere i concetti architetturali chiave.

## **0.1 Setup dell'ambiente**

In [37]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Su Windows: decommenta e adatta se winutils non e' gia' nelle variabili di sistema
# os.environ["HADOOP_HOME"] = r"C:\hadoop"
# os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

print("Interprete    :", sys.executable)
print("HADOOP_HOME   :", os.environ.get("HADOOP_HOME", "NON IMPOSTATO"))
print("SPARK_LOCAL_IP:", os.environ.get("SPARK_LOCAL_IP"))

Interprete    : C:\Users\Allysa\Documents\MachineLearning-in-Spark\.venv\Scripts\python.exe
HADOOP_HOME   : C:\hadoop
SPARK_LOCAL_IP: 127.0.0.1


## **1. Avvio della sessione Spark**

In [38]:
from pyspark.sql import SparkSession

spark = (SparkSession
    .builder
    .appName("IrisPreProcessing")
    .config("master", "local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.python,authenticate.socketTimeout", "60")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.executor.memory", "4g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("Spark avviato:", spark.version)

Spark avviato: 4.2.0


In [39]:
smoke_test_canale = spark.sparkContext.parallelize([1, 2, 3]).map(lambda x: x * 2).collect()

if smoke_test_canale == [2, 4, 6]:
    print("Test superato.")
else:
    raise ValueError("Errore nella comucazione JVM driver <-> Python worker")

Test superato.


## **2. Caricamento dati**

In [40]:
DATA_PATH = Path("data/iris.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"File non esiste nel percorso specificato")

In [41]:
#Con schema implicito
df = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- species: string (nullable = true)

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows


In [42]:
#Con schema esplicito (User-defined)
from pyspark.sql.types import StructField, StructType, StringType, DoubleType

column_mapping = {
    "sepal_length": DoubleType(),
    "sepal_width": DoubleType(),
    "petal_length": DoubleType(),
    "petal_width": DoubleType(),
    "species": StringType(),
}

SCHEMA = StructType([
    StructField(name, dtype, True) for name, dtype in column_mapping.items()
])

df = spark.read.option("header", "true").csv(str(DATA_PATH), schema=SCHEMA).cache()

df.show(5)

#Conteggio per specie
df.groupBy("species").count().show()

+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 5 rows
+----------+-----+
|   species|count|
+----------+-----+
|versicolor|   50|
| virginica|   50|
|    setosa|   50|
+----------+-----+



## **3. Statistiche descrittive**

> Le colonne derivate `sepal_area` e `petal_area` sono state create esclusivamente come feature sintetiche. Non esiste un calcolo analogo per i sepali o i petali.

In [43]:
import pyspark.sql.functions as F

df_area = (
    df.withColumn(("sepal_area"), F.col("sepal_length") * F.col("sepal_width"))
      .withColumn(("petal_area"), F.col("petal_length") * F.col("petal_width"))
)

stats_sepal= df_area.agg(
    F.round(F.mean("sepal_length")).alias("media_length"),
    F.round(F.mean("sepal_width")).alias("media_width"),
    F.round(F.mean("sepal_area")).alias("media_area"),
    F.min("sepal_area").alias("min_area"),
    F.max("sepal_area").alias("max_area")
)

stats_petal= df_area.agg(
    F.round(F.mean("petal_length")).alias("media_length"),
    F.round(F.mean("petal_width")).alias("media_width"),
    F.round(F.mean("petal_area")).alias("media_area"),
    F.round(F.min("petal_area")).alias("min_area"),
    F.max("petal_area").alias("max_area")
)

print("======================== SEPAL ========================")
stats_sepal.show()
print("======================== PETAL ========================")
stats_petal.show()

======================== SEPAL ========================
+------------+-----------+----------+--------+--------+
|media_length|media_width|media_area|min_area|max_area|
+------------+-----------+----------+--------+--------+
|         6.0|        3.0|      18.0|    10.0|   30.02|
+------------+-----------+----------+--------+--------+

======================== PETAL ========================
+------------+-----------+----------+--------+--------+
|media_length|media_width|media_area|min_area|max_area|
+------------+-----------+----------+--------+--------+
|         4.0|        1.0|       6.0|     0.0|   15.87|
+------------+-----------+----------+--------+--------+



In [44]:
#Media della lunghezza e grandezza del petalo e sepalo per specie
df_aggregati = df.groupBy("species").agg(
    F.count("species").alias("count"),
    F.round(F.mean("petal_width")).alias("petal_width"),
    F.round(F.mean("petal_length")).alias("petal_length"),
    F.round(F.mean("sepal_width")).alias("sepal_width"),
    F.round(F.mean("sepal_length")).alias("sepal_length"),
)

df_aggregati.show()

+----------+-----+-----------+------------+-----------+------------+
|   species|count|petal_width|petal_length|sepal_width|sepal_length|
+----------+-----+-----------+------------+-----------+------------+
|versicolor|   50|        1.0|         4.0|        3.0|         6.0|
| virginica|   50|        2.0|         6.0|        3.0|         7.0|
|    setosa|   50|        0.0|         1.0|        3.0|         5.0|
+----------+-----+-----------+------------+-----------+------------+



In [45]:
#Operazioni filtro sepal/petal area > sepal/petal area media

val_scalare_sepal = df_area.select(F.mean("sepal_area")).collect()[0][0]
sepal_filtro = df_area.filter(F.col("sepal_area") > val_scalare_sepal)

val_scalare_petal = df_area.select(F.mean("petal_area")).collect()[0][0]
petal_filtro = df_area.filter(F.col("petal_area") > val_scalare_sepal)

print(f"Sepal area > media: {sepal_filtro.count()}")
print(f"Petal area > media: {petal_filtro.count()}")

Sepal area > media: 71
Petal area > media: 0


## **4. Introduzione al Machine Learning con Pyspark**

> 1. Codifica della colonna `species` mediante **StringIndexer**
> 2. Conversione indici numerici-stringa (round-trip) tramite **IndexToString** per verificare eventuali discordanze
> 3. Costruizione di una Pipeline con tutti gli stage in sequenza

> **StringIndexer** è di tipo Estimator: necessita di analizzare prima i dati tramite il metodo *.fit()* per apprendere i parametri. Al contrario, **IndexToString** è un Transformer, quindi applica direttamente la trasformazione (.transform()) restituendo un nuovo DataFrame.

In [46]:
from pyspark.ml.feature import StringIndexer, IndexToString, VectorAssembler
from pyspark.ml import Pipeline

#Stringhe -> Indici numerici
indexer = StringIndexer(
    inputCol="species",
    outputCol="species_idx",
    stringOrderType="alphabetAsc"
)

indexer_model = indexer.fit(df)
indexed_df = indexer_model.transform(df)
indexed_df.select("species","species_idx").distinct().show()

+----------+-----------+
|   species|species_idx|
+----------+-----------+
|    setosa|        0.0|
|versicolor|        1.0|
| virginica|        2.0|
+----------+-----------+



In [47]:
#Unire le 4 colonne (features) in un unico vettore
features = ["sepal_length", "petal_length", "sepal_width", "petal_width"]
assembler = VectorAssembler(
    inputCols=features,
    outputCol="features",
)

#Indici numerici -> Stringhe
converter = IndexToString(
    inputCol="species_idx",
    outputCol="species_ricostruita",
    labels=indexer_model.labels
)

ricostruita_df = converter.transform(indexed_df)
ricostruita_df.select("species","species_idx","species_ricostruita").distinct().show()

verifica_discordanze = ricostruita_df.filter(F.col("species_ricostruita") != F.col("species")).count()
print(f"Numero di discordanze trovate: {verifica_discordanze}")

+----------+-----------+-------------------+
|   species|species_idx|species_ricostruita|
+----------+-----------+-------------------+
|    setosa|        0.0|             setosa|
| virginica|        2.0|          virginica|
|versicolor|        1.0|         versicolor|
+----------+-----------+-------------------+

Numero di discordanze trovate: 0


In [48]:
#Costruzione della Pipeline
pipeline = Pipeline(
    stages=[indexer, assembler, converter]
)

pipeline_model = pipeline.fit(df)
df_trasformato = pipeline_model.transform(df)

df_trasformato.select("species", "species_idx", "features", "species_ricostruita").show(5, truncate=False)

+-------+-----------+-----------------+-------------------+
|species|species_idx|features         |species_ricostruita|
+-------+-----------+-----------------+-------------------+
|setosa |0.0        |[5.1,1.4,3.5,0.2]|setosa             |
|setosa |0.0        |[4.9,1.4,3.0,0.2]|setosa             |
|setosa |0.0        |[4.7,1.3,3.2,0.2]|setosa             |
|setosa |0.0        |[4.6,1.5,3.1,0.2]|setosa             |
|setosa |0.0        |[5.0,1.4,3.6,0.2]|setosa             |
+-------+-----------+-----------------+-------------------+
only showing top 5 rows


## **5. Confronto tra UDF Pyspark e funzioni native**
> Questa cella ha l'obiettivo di valutare la differenza tra l'esecuzione via **UDF (User-Defined Function)** Python (che comporta costi di serializzazione) e l'ottimizzazione garantita dal **Catalyst Optimizer** con le funzioni native Spark.

> Inoltre, va a verificare la correttezza dei risultati attraverso il metodo *.eqNullSafe()*

> **Scopo**: Definire una funzione che etichetta il fiore come grande se `petal_width` > 1.5, altrimenti come piccolo. Viene gestito anche il caso in cui il valore in ingresso sia *None*.

In [49]:
def iris_classificatore(petal_width):
    if petal_width >= 1.5:
        return "grande"
    elif None:
        return None
    else:
        return "piccolo"

fiore_grandezza = F.udf(iris_classificatore, StringType())
df_classificato = df.withColumn("class_udf", fiore_grandezza("petal_width"))

df_classificato.select("petal_width","class_udf").orderBy(F.rand()).show(5)


+-----------+---------+
|petal_width|class_udf|
+-----------+---------+
|        1.8|   grande|
|        2.3|   grande|
|        0.2|  piccolo|
|        1.6|   grande|
|        1.2|  piccolo|
+-----------+---------+
only showing top 5 rows


In [50]:
df_classificato = df_classificato.withColumn("class_nativo",
                                             F.when(F.col("petal_width").isNull(), None)
                                             .when(F.col("petal_width") >= 1.5, "grande")
                                             .otherwise("piccolo"))

df_classificato.select("petal_width","class_udf","class_nativo").orderBy(F.rand()).show(5)

+-----------+---------+------------+
|petal_width|class_udf|class_nativo|
+-----------+---------+------------+
|        0.4|  piccolo|     piccolo|
|        1.3|  piccolo|     piccolo|
|        1.9|   grande|      grande|
|        0.2|  piccolo|     piccolo|
|        2.0|   grande|      grande|
+-----------+---------+------------+
only showing top 5 rows


In [51]:
df_verifica_equivalenza = df_classificato.withColumn("coincidono",
                                                   F.col("class_udf").eqNullSafe(F.col("class_nativo")))

totale_record = df_classificato.count()
totale_coincidono = df_verifica_equivalenza.filter(F.col("coincidono") == True).count()
print(f"Numero totale del record: {totale_record}\n"
      f"Numero totale coincidono: {totale_coincidono}")

Numero totale del record: 150
Numero totale coincidono: 150


In [52]:
#Tabella incrociata species x classificazione della dimensione
df_classificato.crosstab("species", "class_udf").show()

+-----------------+------+-------+
|species_class_udf|grande|piccolo|
+-----------------+------+-------+
|       versicolor|    15|     35|
|        virginica|    49|      1|
|           setosa|     0|     50|
+-----------------+------+-------+



In [53]:
spark.stop()